In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

from data.data_loader import get_data_loaders
from models.mnist import mnist
from utils.training import train, evaluate
from utils.metrics import calculate_sparsity, count_unique_weights

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, test_loader = get_data_loaders()

model = mnist().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

100%|██████████| 9.91M/9.91M [00:07<00:00, 1.24MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 118kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 798kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.71MB/s]


In [4]:
for epoch in range(5):
    loss = train(model, train_loader, optimizer, criterion, device)
    acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={acc:.2f}%")

baseline_acc = evaluate(model, test_loader, device)

Epoch 1: Loss=0.4355, Acc=94.86%
Epoch 2: Loss=0.1396, Acc=96.88%
Epoch 3: Loss=0.0938, Acc=96.78%
Epoch 4: Loss=0.0670, Acc=97.46%
Epoch 5: Loss=0.0510, Acc=97.29%


In [5]:
PRUNE_RATIO = 0.5
model.prune(PRUNE_RATIO)

sparsity = calculate_sparsity(model)
print(f"Sparsity: {sparsity:.2f}%")

pruned_linear!!!
pruned_linear!!!
pruned_linear!!!
pruned_linear!!!
Sparsity: 50.00%


In [6]:
for epoch in range(3):
    loss = train(model, train_loader, optimizer, criterion, device)
    acc = evaluate(model, test_loader, device)
    print(f"[Finetune] Epoch {epoch+1}: Loss={loss:.4f}, Acc={acc:.2f}%")

pruned_acc = evaluate(model, test_loader, device)

[Finetune] Epoch 1: Loss=0.0306, Acc=97.90%
[Finetune] Epoch 2: Loss=0.0229, Acc=97.95%
[Finetune] Epoch 3: Loss=0.0191, Acc=98.01%


In [7]:
NUM_BITS = 8
model.quantize(NUM_BITS)

quant_acc = evaluate(model, test_loader, device)
unique_vals = count_unique_weights(model)

print(f"Quantized Accuracy: {quant_acc:.2f}%")
print(f"Unique Weights: {unique_vals}")

quantized_linear!!
quantized_linear!!
quantized_linear!!
quantized_linear!!
Quantized Accuracy: 98.00%
Unique Weights: 772
